# Market-Basket Association Mining

This notebook audits basket structure, executes the rule-mining workflow, compares the two implementations, and reconciles the selected results with exported artifacts.

## 1. Setup and transaction audit

The transactions are synthetic and the analysis treats rules as co-occurrence, not preference or causation.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'transactions.json'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts'
baskets = json.loads(DATA_PATH.read_text())
audit = {'transactions': len(baskets), 'unique_items': len(set(item for basket in baskets for item in basket)), 'average_basket_size': round(sum(map(len, baskets)) / len(baskets), 4), 'empty_baskets': sum(not basket for basket in baskets), 'duplicate_items': sum(len(basket) != len(set(basket)) for basket in baskets)}
audit

## 2. Execute the reproducible experiment

The command applies the documented support threshold and exports Apriori-style rules, ECLAT reference rules, algorithm comparison metrics, and a top-rule figure.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, str(PROJECT_ROOT / 'src' / 'run_experiment.py'), '--data', str(DATA_PATH), '--output', str(OUTPUT_DIR)], check=True)

## 3. Rule comparison and visual evidence

Support, confidence, lift, and runtime are reviewed together. High lift from a small or synthetic transaction population should not be treated as a causal business conclusion.

In [ ]:
apriori = pd.read_csv(OUTPUT_DIR / 'apriori-rules.csv')
algorithm_comparison = pd.read_csv(OUTPUT_DIR / 'algorithm-comparison.csv')
display(algorithm_comparison)
display(apriori.head(10))
display(Image(filename=str(OUTPUT_DIR / 'top-rules.png')))

## 4. Reconcile outputs

The JSON summary is authoritative for thresholds, rule counts, top lift, artifact names, and limitations.

In [ ]:
summary = json.loads((OUTPUT_DIR / 'results-summary.json').read_text())
assert summary['metrics']['rules'] == len(apriori)
summary

## Interpretation

Association mining identifies items appearing together. Any recommendation would require support stability checks, seasonality review, product-availability checks, privacy review, and controlled testing.